# Examine drop-out risks and protective factors
We will use a combination of 
1. XGBoost global feature importance
2. Permutation importance
3. SHAP global summary

and compare the results to derive insights on drop-out risks and protective factors. We combine three complementary methods to ensure robustness and validity of our findings.

Note that none of these methods provide causal insights, but they can help *identify important factors* associated with student drop-out.

In [ ]:
# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
import shap
import warnings
warnings.filterwarnings('ignore')
import os
from pathlib import Path

print("Libraries imported successfully!")

In [ ]:
# Load the tuned model
current_user = os.getenv('USER')
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")
final_model = pd.read_pickle(PROCESSED_DIR / "xgb_final_tuned_trainval.pkl")

print(f"Model loaded successfully!")

## 1. XGBoost Global Feature Importance

Global feature importance shows which features are most important for the model's decisions across all predictions. We'll use the 'gain' metric, which measures the average improvement in model performance when a feature is used for splitting.

In [ ]:
# Extract XGBoost global feature importance
# Get importance scores for different metrics
importance_gain = final_model.get_booster().get_score(importance_type='gain')
importance_weight = final_model.get_booster().get_score(importance_type='weight')
importance_cover = final_model.get_booster().get_score(importance_type='cover')

# Convert to DataFrame for easier analysis
def importance_to_df(importance_dict, importance_type):
    df = pd.DataFrame(list(importance_dict.items()), 
                     columns=['feature', f'importance_{importance_type}'])
    df = df.sort_values(f'importance_{importance_type}', ascending=False)
    return df

df_gain = importance_to_df(importance_gain, 'gain')
df_weight = importance_to_df(importance_weight, 'weight')
df_cover = importance_to_df(importance_cover, 'cover')

In [ ]:
# Visualize XGBoost global feature importance
plt.figure(figsize=(12, 12))

# Plot top 20 features by gain
top_20_gain = df_gain.head(20)
plt.subplot(3, 1, 1)
plt.barh(range(len(top_20_gain)), top_20_gain['importance_gain'])
plt.yticks(range(len(top_20_gain)), top_20_gain['feature'])
plt.xlabel('Importance Score (Gain)')
plt.title('Top 20 Features - XGBoost Global Feature Importance (Gain)')
plt.gca().invert_yaxis()

# Plot top 20 features by weight  
top_20_weight = df_weight.head(20)
plt.subplot(3, 1, 2)
plt.barh(range(len(top_20_weight)), top_20_weight['importance_weight'])
plt.yticks(range(len(top_20_weight)), top_20_weight['feature'])
plt.xlabel('Importance Score (Weight)')
plt.title('Top 20 Features - XGBoost Global Feature Importance (Weight)')
plt.gca().invert_yaxis()

# Plot top 20 features by cover  
top_20_cover = df_cover.head(20)
plt.subplot(3, 1, 3)
plt.barh(range(len(top_20_cover)), top_20_cover['importance_cover'])
plt.yticks(range(len(top_20_cover)), top_20_cover['feature'])
plt.xlabel('Importance Score (Cover)')
plt.title('Top 20 Features - XGBoost Global Feature Importance (Cover)')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

# Print summary statistics
print("\nFeature Importance Summary (Gain):")
print(f"Mean importance: {df_gain['importance_gain'].mean():.4f}")
print(f"Median importance: {df_gain['importance_gain'].median():.4f}")
print(f"Standard deviation: {df_gain['importance_gain'].std():.4f}")
print(f"Top feature importance: {df_gain['importance_gain'].max():.4f}")
print(f"Importance ratio (top/median): {df_gain['importance_gain'].max() / df_gain['importance_gain'].median():.2f}")

## 1.1 Results & interpretation
We extract the global feature importance from the trained XGBoost model and display the top features based on the `gain` metric, as well as the `weight` metric. A high `gain` indicates that the feature improves the model's predictive power when used for splits: the model becomes better at distinguishing between drop-outs and non-drop-outs. A high `weight` indicates that the feature is frequently used for splits in the decision trees. For complementary information we also assess `cover`, which measures the relative coverage of samples affected by splits on a feature (i.e. a feature's influence). 

Overall (based on both gain and weight), the most important features are course result related: `Total Credits Block A`, `Average Grade A` and `Potential Credits A Missing Flag`. This suggests that students' academic performance in Block A is a key factor associated with drop-out risk. Note that the feature `Potential Credits A Missing Flag` does not have a high weight, indicating that it might only be important for a subset of students.

Examining the top feature by `cover`, we see a lot of degrees appearing. This tells us that the model segments students by degree, indicating that the model considers degree-level differences important when predicting drop-out risk: different degree programs have distinct dropout patterns. 

## 2. Permutation Importance

Permutation importance shows how much the model's performance decreases when each feature is randomly shuffled. This gives us a more robust measure of feature importance that's less biased toward high-cardinality features.

## 3. SHAP Global Summary

SHAP (SHapley Additive exPlanations) values provide a unified framework for interpreting model predictions. We'll create a global summary showing the average impact of each feature.

## 4. Comparative Analysis

Let's compare the results from all three methods to identify the most robust dropout risk factors.